N7023-SEMCD-14_UdayKiranMudu

## 1. ENVIRONMENT SETUP & SPARK INITIALIZATION

In [ ]:
!pip install pyspark -q

In [ ]:
import requests
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Spambase_Perceptron_Classification") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Successfully Created!")

Spark Session Successfully Created!


## 2. DATA LOADING & PREPARATION

In [ ]:
# Download Spambase Dataset from UCI ML Repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"
response = requests.get(url)
with open("spambase.data", "wb") as f:
    f.write(response.content)

In [ ]:
# Define column names (57 features + 1 target label column)
feature_cols = [f"word_freq_{i}" for i in range(1, 49)] + \
               [f"char_freq_{i}" for i in range(1, 7)] + \
               ["capital_run_length_average", "capital_run_length_longest", "capital_run_length_total"]
schema_cols = feature_cols + ["label"]

# Load dataset into Spark DataFrame
df = spark.read.csv("spambase.data", inferSchema=True, header=False)
for old_col, new_col in zip(df.columns, schema_cols):
    df = df.withColumnRenamed(old_col, new_col)

# Cast label to double
df = df.withColumn("label", col("label").cast("double"))

## 3. DATA EXPLORATION

In [ ]:
print(f"Total Records: {df.count()}")
print(f"Total Features: {len(feature_cols)}")
df.groupBy("label").count().show()

Total Records: 4601
Total Features: 57
+-----+-----+
|label|count|
+-----+-----+
|  0.0| 2788|
|  1.0| 1813|
+-----+-----+



## 4. FEATURE PIPELINE & SCALING

In [ ]:
# Assemble features into a single dense vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
assembled_df = assembler.transform(df)

In [ ]:
# Feature Scaling: Standardize features (mean=0, std=1)
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(assembled_df)
scaled_df = scaler_model.transform(assembled_df)

In [ ]:
# Train-Test Split (80% Train, 20% Test)
train_df, test_df = scaled_df.select("scaled_features", "label").randomSplit([0.8, 0.2], seed=42)
print(f"Training set count: {train_df.count()}, Testing set count: {test_df.count()}")

Training set count: 3725, Testing set count: 876


## 5. PERCEPTRON ALGORITHM IMPLEMENTATION (RDD-based Distributed Training)

In [ ]:
# Convert DataFrames to RDD for custom weight updates: (label, numpy array of features)
train_rdd = train_df.rdd.map(lambda row: (row["label"], row["scaled_features"].toArray())).cache()
test_rdd = test_df.rdd.map(lambda row: (row["label"], row["scaled_features"].toArray())).cache()

In [ ]:
def train_perceptron(rdd, num_features, lr=0.01, epochs=20):
    """
    Trains a Perceptron binary classifier over Spark RDD dataset.
    Weights are updated iteratively based on misclassifications.
    """
    weights = np.zeros(num_features)
    bias = 0.0

    for epoch in range(1, epochs + 1):
        # Broadcast current weights and bias across cluster
        b_weights = spark.sparkContext.broadcast(weights)
        b_bias = spark.sparkContext.broadcast(bias)

        # Identify misclassified samples and compute weight updates
        def get_gradient(record):
            label, x = record
            w = b_weights.value
            b = b_bias.value
            # Perceptron activation step: sign(w . x + b)
            activation = np.dot(w, x) + b
            pred = 1.0 if activation >= 0.0 else 0.0
            error = label - pred  # (y - y_hat)
            if error != 0:
                return (error * x, error)
            else:
                return (np.zeros_like(x), 0.0)

        # Aggregate weight updates across partitions
        grad_w, grad_b = rdd.map(get_gradient).reduce(
            lambda a, b: (a[0] + b[0], a[1] + b[1])
        )

        # Apply learning rate updates
        weights += lr * grad_w
        bias += lr * grad_b

        # Cleanup broadcast variables
        b_weights.unpersist()
        b_bias.unpersist()

    return weights, bias

In [ ]:
# Train the model
num_features = len(feature_cols)
weights, bias = train_perceptron(train_rdd, num_features=num_features, lr=0.005, epochs=30)
print("\nModel Training Completed!")


Model Training Completed!


## 6. PREDICTION & INFERENCE

In [ ]:
b_weights_final = spark.sparkContext.broadcast(weights)
b_bias_final = spark.sparkContext.broadcast(bias)

def predict(x):
    activation = np.dot(b_weights_final.value, x) + b_bias_final.value
    return 1.0 if activation >= 0.0 else 0.0

In [ ]:
# Generate predictions on Test RDD
predictions_rdd = test_rdd.map(lambda row: (row[0], float(predict(row[1]))))

In [ ]:
# Convert predictions back to Spark DataFrame for metric evaluation
predictions_df = spark.createDataFrame(predictions_rdd, ["label", "prediction"])

## 7. MODEL EVALUATION & METRICS PERFORMANCE

In [ ]:
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

accuracy = evaluator_acc.evaluate(predictions_df)
precision = evaluator_prec.evaluate(predictions_df)
recall = evaluator_rec.evaluate(predictions_df)
f1_score = evaluator_f1.evaluate(predictions_df)

print("\n" + "="*40)
print("       PERCEPTRON EVALUATION METRICS     ")
print("="*40)
print(f" Accuracy  : {accuracy * 100:.2f}%")
print(f" Precision : {precision * 100:.2f}%")
print(f" Recall    : {recall * 100:.2f}%")
print(f" F1-Score  : {f1_score * 100:.2f}%")
print("="*40)


       PERCEPTRON EVALUATION METRICS     
 Accuracy  : 86.99%
 Precision : 88.93%
 Recall    : 86.99%
 F1-Score  : 87.04%
